# Open-data explanation-quality benchmark

Run the explanation-quality benchmark on the open clinical datasets.

Outputs (one set per dataset) go to OUTPUT_DIR/results/<dataset>/.

## Setup
Loads `00_config` (paths) and the library notebooks from this folder.

In [7]:
# Load other notebooks in this folder as modules (all notebooks live in one folder).
import os, sys, types
import nbformat

NB_DIR = os.getcwd()   # open Jupyter in the notebooks folder


def import_notebook(name, filename):
    """Execute the code cells of a notebook in this folder into a module named `name`."""
    if name in sys.modules:
        return sys.modules[name]
    path = os.path.join(NB_DIR, filename)
    mod = types.ModuleType(name)
    mod.__file__ = path
    sys.modules[name] = mod
    for cell in nbformat.read(path, as_version=4).cells:
        if cell.cell_type == "code":
            exec(compile(cell.source, path, "exec"), mod.__dict__)
    return mod


config = import_notebook("config", "00_config.ipynb")

In [8]:
data = import_notebook("clinxai.data", "01_data_loaders.ipynb")
core = import_notebook("clinxai.core", "02_core_methods.ipynb")

In [9]:
import json
import os
import sys
import time
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.model_selection import train_test_split
OUT = config.RESULTS_DIR
MODELS = ["LR", "EBM", "XGB", "MLP"]
N_EXPLAIN = 80
N_BG = 50
N_PERTURB = 3
N_RETRAIN = 2
N_REPEAT = 5
ROAR_K = [1, 3, 5]
ROAR_MODELS = ["LR", "XGB"]

### `boot_ci` (function)

In [10]:
def boot_ci(y, p, fn, n=1000, seed=0):
    rng = np.random.default_rng(seed)
    vals = []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        if y[i].min() == y[i].max():
            continue
        vals.append(fn(y[i], p[i]))
    return np.percentile(vals, [2.5, 97.5])

### `run_dataset` (function)

In [11]:
def run_dataset(name, seed=0):
    t0 = time.time()
    od = os.path.join(OUT, name)
    os.makedirs(od, exist_ok=True)
    d = data.load(name)
    idx = np.arange(len(d.y))
    tr, te = train_test_split(idx, test_size=0.3, stratify=d.y, random_state=seed)
    enc = core.Encoder().fit(d.X.iloc[tr], d.cat)
    Ztr, Zte = enc.transform(d.X.iloc[tr]), enc.transform(d.X.iloc[te])
    ytr, yte = d.y[tr], d.y[te]
    base = enc.baseline(Ztr)
    rng = np.random.default_rng(seed)
    Zbg = Ztr[rng.choice(len(Ztr), min(N_BG, len(Ztr)), replace=False)]
    nx = min(N_EXPLAIN, len(te))
    ex_pos, _ = train_test_split(np.arange(len(te)), train_size=nx, stratify=yte, random_state=seed) \
        if nx < len(te) else (np.arange(len(te)), None)
    Zx = Zte[ex_pos]
    grp = d.group.iloc[te].values[ex_pos] if d.group is not None else np.array([None] * nx)

    perf, inst, agree, roar, timing, glob = [], [], [], [], [], []
    params_all, attrs = {}, {}
    for kind in MODELS:
        tm = time.time()
        if len(Ztr) > 15000:  # large cohorts: tune on a stratified 10,000-row subsample of the training split
            sub, _ = train_test_split(np.arange(len(Ztr)), train_size=10000, stratify=ytr, random_state=seed)
            params = core.tune(kind, enc, Ztr[sub], ytr[sub], seed=seed, n_iter=8)
        else:
            params = core.tune(kind, enc, Ztr, ytr, seed=seed)
        params_all[kind] = {k: (list(v) if isinstance(v, tuple) else v) for k, v in params.items()}
        model = core.make_model(kind, enc, seed, params).fit(Ztr, ytr)
        fit_time = time.time() - tm
        f = core.proba_fn(model)
        p = f(Zte)
        auc_ci = boot_ci(yte, p, roc_auc_score)
        # repeated splits (hyper-parameters fixed from split 0)
        rep = []
        for r in range(1, N_REPEAT):
            tr2, te2 = train_test_split(idx, test_size=0.3, stratify=d.y, random_state=r)
            e2 = core.Encoder().fit(d.X.iloc[tr2], d.cat)
            m2 = core.make_model(kind, e2, r, params).fit(e2.transform(d.X.iloc[tr2]), d.y[tr2])
            rep.append(roc_auc_score(d.y[te2], m2.predict_proba(e2.transform(d.X.iloc[te2]))[:, 1]))
        perf.append(dict(dataset=name, model=kind, auroc=roc_auc_score(yte, p), auroc_lo=auc_ci[0],
                         auroc_hi=auc_ci[1], auprc=average_precision_score(yte, p), brier=brier_score_loss(yte, p),
                         prevalence=yte.mean(), auroc_rep_mean=np.mean([roc_auc_score(yte, p)] + rep),
                         auroc_rep_sd=np.std([roc_auc_score(yte, p)] + rep, ddof=1), fit_time=fit_time,
                         n_train=len(tr), n_test=len(te)))

        # ---- explanations on the explanation set
        res, t = core.explain_all(kind, model, enc, Ztr, Zbg, Zx, seed=seed)
        rand_list = [core.explain_random(Zx, s) for s in range(5)]
        for a, v in t.items():
            timing.append(dict(dataset=name, model=kind, explainer=a, seconds=v, n=nx, per_instance=v / nx))
        expl_names = [a for a in res if a != "Random"]
        for a in expl_names:
            attrs[f"{kind}|{a}"] = res[a]

        # ---- deletion faithfulness
        rows = {}
        for a in expl_names:
            mo, le = core.deletion_curves(f, Zx, res[a], base)
            rows[a] = dict(morf=mo, lerf=le, ent=core.entropy(res[a]))
        rmo, rle = zip(*[core.deletion_curves(f, Zx, R, base) for R in rand_list])
        rows["Random"] = dict(morf=np.mean(rmo, axis=0), lerf=np.mean(rle, axis=0),
                              ent=np.mean([core.entropy(R) for R in rand_list], axis=0))

        # ---- input-perturbation stability
        pert_sp = {a: [] for a in expl_names}
        pert_jac = {a: [] for a in expl_names}
        dp = []
        for k in range(N_PERTURB):
            Zp = core.perturb(Zx, enc, Ztr, 0.05, np.random.default_rng(100 + k))
            dp.append(np.abs(f(Zp) - f(Zx)))
            rp, _ = core.explain_all(kind, model, enc, Ztr, Zbg, Zp, seed=seed,
                                     which=["KernelSHAP", "LIME", "native"])
            for a in expl_names:
                s, j = core.stability(res[a], rp[a])
                pert_sp[a].append(s); pert_jac[a].append(j)
        # ---- retraining stability (bootstrap refits with fixed hyper-parameters)
        rt_sp = {a: [] for a in expl_names}
        rt_jac = {a: [] for a in expl_names}
        for k in range(N_RETRAIN):
            bi = np.random.default_rng(200 + k).integers(0, len(Ztr), len(Ztr))
            mk = core.make_model(kind, enc, 200 + k, params).fit(Ztr[bi], ytr[bi])
            rk, _ = core.explain_all(kind, mk, enc, Ztr, Zbg, Zx, seed=seed, which=["KernelSHAP", "LIME", "native"])
            for a in expl_names:
                s, j = core.stability(res[a], rk[a])
                rt_sp[a].append(s); rt_jac[a].append(j)

        for a in list(expl_names) + ["Random"]:
            for i in range(nx):
                row = dict(dataset=name, model=kind, explainer=a, inst=int(ex_pos[i]), group=grp[i], y=int(yte[ex_pos[i]]),
                           morf=rows[a]["morf"][i], lerf=rows[a]["lerf"][i], entropy=rows[a]["ent"][i])
                if a != "Random":
                    row.update(pert_sp=np.mean([s[i] for s in pert_sp[a]]),
                               pert_jac=np.mean([s[i] for s in pert_jac[a]]),
                               retrain_sp=np.mean([s[i] for s in rt_sp[a]]),
                               retrain_jac=np.mean([s[i] for s in rt_jac[a]]),
                               pert_dp=np.mean([x[i] for x in dp]))
                inst.append(row)

        # ---- agreement between explainers
        for i1, a in enumerate(expl_names):
            for b in expl_names[i1 + 1:]:
                fa, sa, rc = core.pairwise_agreement(res[a], res[b], k=5)
                agree.append(dict(dataset=name, model=kind, pair=f"{a} vs {b}", a=a, b=b,
                                  feat_agree=fa.mean(), sign_agree=sa.mean(), rank_corr=rc.mean(),
                                  feat_agree_sd=fa.std(), rank_corr_sd=rc.std()))

        # ---- global importance
        for a in expl_names:
            g = np.abs(res[a]).mean(axis=0)
            for j, c in enumerate(enc.cols):
                glob.append(dict(dataset=name, model=kind, explainer=a, feature=c, importance=g[j],
                                 rank=int((-g).argsort().argsort()[j]) + 1))

        # ---- ROAR (remove and retrain)
        if kind in ROAR_MODELS:
            full = roc_auc_score(yte, p)
            rankings = {a: np.argsort(-np.abs(res[a]).mean(axis=0)) for a in expl_names}
            for s in range(5):
                rankings[f"Random#{s}"] = np.random.default_rng(300 + s).permutation(Ztr.shape[1])
            for a, order in rankings.items():
                for k in ROAR_K:
                    if k >= Ztr.shape[1]:
                        continue
                    drop = order[:k]
                    Ztr_k, Zte_k = Ztr.copy(), Zte.copy()
                    Ztr_k[:, drop] = base[drop]
                    Zte_k[:, drop] = base[drop]
                    mk = core.make_model(kind, enc, seed, params).fit(Ztr_k, ytr)
                    roar.append(dict(dataset=name, model=kind, explainer=a.split("#")[0], k=k,
                                     auroc=roc_auc_score(yte, mk.predict_proba(Zte_k)[:, 1]), auroc_full=full))
        print(f"[{name}] {kind} done in {time.time() - tm:.0f}s (AUROC {perf[-1]['auroc']:.3f})", flush=True)

    pd.DataFrame(perf).to_csv(os.path.join(od, "performance.csv"), index=False)
    pd.DataFrame(inst).to_csv(os.path.join(od, "instance_metrics.csv"), index=False)
    pd.DataFrame(agree).to_csv(os.path.join(od, "agreement.csv"), index=False)
    pd.DataFrame(roar).to_csv(os.path.join(od, "roar.csv"), index=False)
    pd.DataFrame(timing).to_csv(os.path.join(od, "timing.csv"), index=False)
    pd.DataFrame(glob).to_csv(os.path.join(od, "global_importance.csv"), index=False)
    np.savez_compressed(os.path.join(od, "attributions.npz"), Zx=Zx, cols=np.array(enc.cols), **attrs)
    meta = dict(d.meta, n=len(d.y), n_features=len(enc.cols), n_categorical=len(enc.cat_idx),
                prevalence=float(d.y.mean()), missing_rate=float(d.X.isna().mean().mean()),
                n_explain=int(nx), params=params_all, runtime_s=time.time() - t0,
                max_abs_corr=float(np.nanmax(np.abs(np.triu(np.corrcoef(Ztr.T), 1)))))
    json.dump(meta, open(os.path.join(od, "meta.json"), "w"), indent=1, default=str)
    print(f"[{name}] finished in {time.time() - t0:.0f}s", flush=True)

## Run
All seven datasets take about 1 hour on 2 CPU cores (Diabetes 130 is the slowest). Use e.g. `DATASETS = ["Heart"]` for a quick test.

In [12]:
DATASETS = list(data.LOADERS)
for name in DATASETS:
    run_dataset(name)

[CKD] reading D:\New_Research_2027\Clinical_XAI_Faithfulness\uci_downloads\uci_336_chronic_kidney_disease.csv
[CKD] LR done in 56s (AUROC 1.000)
[CKD] EBM done in 77s (AUROC 1.000)
[CKD] XGB done in 35s (AUROC 1.000)
[CKD] MLP done in 72s (AUROC 0.999)
[CKD] finished in 239s
[Heart] reading D:\New_Research_2027\Clinical_XAI_Faithfulness\heart+disease\processed.cleveland.data
[Heart] LR done in 40s (AUROC 0.893)
[Heart] EBM done in 41s (AUROC 0.893)
[Heart] XGB done in 29s (AUROC 0.896)
[SUPPORT2] MLP done in 111s (AUROC 0.785)
[SUPPORT2] finished in 764s
[Diabetes130] reading D:\New_Research_2027\Clinical_XAI_Faithfulness\diabetes+130-us+hospitals+for+years+1999-2008\diabetic_data.csv
[Diabetes130] LR done in 72s (AUROC 0.589)
[Diabetes130] EBM done in 187s (AUROC 0.593)
[Diabetes130] XGB done in 45s (AUROC 0.593)
[Diabetes130] MLP done in 452s (AUROC 0.581)
[Diabetes130] finished in 757s
